* This script is designed to interact with the ODR server and provides functionality to retrieve records, create new records, delete existing records, upload files to a record, and perform other helpful operations.
* It also serves as a reference for understanding how records are structured within the API, which is useful when uploading raw CSV files into the ODR database.
* The script requires a username, password, and server URL for authentication and token generation.
* Once authenticated, the available functions can be used as needed

In [54]:
## If you have any questions about this script, feel free to contact me @ someone@example.gov, Thanks.

In [5]:
# Import required libraries
import requests        # Send HTTP requests to the server
import json            # Convert server responses to Python dictionaries
import os              # File path handling and OS-level checks
import time            # Token expiration handling and timing utilities
import jwt             # Decode JWT authentication tokens
from typing import Dict, Optional  # Type hints for arguments and return values
import copy            # Used for copying objects, helpful for header manipulation
import pandas as pd    # Read and process data files (e.g., CSV, Excel)
from pathlib import Path  # Clean, Pythonic handling of filesystem paths

In [6]:
# Global variables
URL = 'https://www.odr.io/api/v4' #API url
USERNAME = 'your_odr_login'   # Replace with your ODR username (contact Nate if you need one)
PASSWORD = 'your_odr_password'   # Replace with the ODR password provided by Nate

# UUID of the SCOBI working repository.
# Note: SCOBI Public and other ODR databases each have their own UUIDs.
DATASET_UUID = '063c0d3d4bd183ab0dda87c544ae'

# Directory where raw data is stored. This should match the BOX folder structure.
# You may also set this to a local path if needed.
DATA_DIR = '/raw/data'

In [7]:
# Main client class with initialization of client and client request methods to the server
class ODR_Client:
    """
    Initialize the instance with the base URL, username, password, token expiration,
    authentication token, and default request headers.
    
    Parameters:
        base_url (str): The API base URL.
        username (str): The user’s login username.
        password (str): The user’s login password.
    
    Returns:
        None
    """
    def __init__(self, base_url: str, username: str, password: str):

        self.username = username
        self.password = password
        self.base_url = base_url
        self.headers = {'Content-Type': 'application/json'}
        self.token_expiry = None
        self.token = None

    """
    Authenticate the user using the provided username and password, and return a
    bearer token for authorized API requests.
    
    This method sends a POST request to the `/token` endpoint, retrieves the
    authentication token, and decodes it with `jwt` (without signature verification).
    It also extracts the token expiration time. If none is provided, a default
    expiration of 3600 seconds (1 hour) is assumed.
    
    Parameters:
        url (str): The API base URL.
        username (str): The user’s login username.
        password (str): The user’s login password.
    
    Returns:
        str: The authentication token.
    """
    def authenticate(self, url : str = None, username: str = None, password: str = None) -> str:
        username = username or self.username
        password = password or self.password
        json = {'password' : password, 'username' : username}
        url = self.base_url or url
        url = f'{url}/token'
        headers = self.headers
        response = requests.post(url, json = json, headers = headers, timeout = 30)
        if response.status_code == 200:
            self.token = response.json().get('token')
            try:
                self.token_expiry = (jwt.decode(self.token, options = {'verify_signature':False})).get('exp', time.time() + 3600)
            except:
                self.token_expiry = time.time() + 3600
            self.headers['Authorization'] = f'Bearer {self.token}'
            #print('Authorization success')
            return self.token
        else:
            print(f'Authentication Failed: {response.status_code}')   

    """
    Method to check whether the authentication token has expired.
    
    Parameters:
        None
    
    Returns:
        bool: True if the token is expired, otherwise False.
    """

    def is_token_expired(self) -> bool:
        if not self.token or self.token_expiry:
            return True
        return time.time() >= self.token_expiry - 30
    
    """
    Helper method to send a request to the server using GET, POST, or other HTTP methods.
    Automatically checks whether the authentication token has expired and refreshes it if needed.
    
    Parameters:
        method (str): The HTTP method to use (e.g., "GET", "POST").
        url (str): The target endpoint for the request.
        **kwargs: Additional options such as headers, JSON payload, form data, etc.
    
    Returns:
        requests.Response: The response object returned by the server.
    """

    def make_request(self, method: str, url: str, **kwargs):
        
        if self.is_token_expired():
            self.authenticate()
        
        response = requests.request(method, url, headers=self.headers, **kwargs)
        
        # If we get a 401, refresh token and try again
        if response.status_code == 401:
            print("🔄 Token expired, refreshing...")
            self.authenticate()
            response = requests.request(method, url, headers=self.headers, **kwargs)
        
        return response
   
    """
    Create an empty record using the `/dataset/{dataset_uuid}/record` endpoint.
    
    This method calls the `make_request` helper with a POST request and a
    120‑second timeout. Record creation may fail if invalid or incomplete
    information is provided.
    
    Parameters:
        dataset_uuid (str): The UUID of the dataset where the record will be created.
        user_email (Optional[str]): The email address of the user creating the record.
    
    Returns:
        dict: The empty record that was created.
    """

    def create_record(self, dataset_uuid : str, user_email: str= None)-> Dict:
        user_email = self.username or user_email
        url = f'{self.base_url}/dataset/{dataset_uuid}/record'
        
        headers = self.headers.copy()
        headers.pop('Content-Type', None)
        
        response = self.make_request('POST', url, timeout=120)

        if response.status_code == 200:
            record = response.json()
            #print(f'Created a record with name: {record.get('record_name')}')
            return record
        else:
             raise Exception(f'failed to create record : {response.status_code} {response.text}')

    """
    Push a record to ODR. Creates and adds fields if the record does not already exist,
    or updates fields if the record is already present.
    
    Parameters:
        record_dict (dict): A dictionary representing the record, including keys such as
            'fields' and 'recorduuid'.
        user_email (str, optional): The email of the user performing the operation.
    
    Returns:
        dict: Information about the record that was pushed.
    """
    def push_record(self, record_dict: Dict, user_email: Optional[str] = None) -> Dict:
        url = f'{self.base_url}/dataset/record'
        if user_email:
            record_dict['user_email'] = user_email
        response = self.make_request('POST', url, json = record_dict, timeout = 120)
        #print(record_dict)
        if response.status_code == 200:
            #print('Record updated')
            return response.json()
        raise RuntimeError(f'failed to update record status code: {response.status_code} and more detail here: {response.text}')
            
    """
    Retrieve a record using its UUID.
    
    This method may return an error if the provided UUID or related information
    is incorrect.
    
    Parameters:
        record_uuid (str): The UUID of the record to retrieve.
    
    Returns:
        dict: The record associated with the given UUID.
    """

    def get_record(self, record_uuid: str) -> Dict:
        url = f'{self.base_url}/dataset/record/{record_uuid}'
    
        response = self.make_request('GET', url, timeout = 120)

        if response.status_code == 200:
            record = response.json()
            print(f'Got a record: {record.get('record_uuid')}')
            return record
        else:
            raise Exception(f'Failed to get a record-- look further at status code: {response.status_code} {response.text}')

    """
    Method to delete a single record from a dataset based on its UUID.
    Raises an error if the provided information is invalid.
    
    Parameters:
        datasetuuid (str): The UUID of the dataset.
        record_uuid (str): The UUID of the record to delete.
        user_email (str, optional): The email of the user performing the deletion; useful for logging.
    
    Returns:
        dict: The deleted record information.
    """

    def delete_record(self, dataset_uuid: str, record_uuid: str, user_email: str = None) -> Dict:
        user_email = user_email or self.username 
        url = f'{self.base_url}/record/{record_uuid}'

        #json = {'record_uuids': record_uuids,
        #       'user_email' : user_email}
        headers = self.headers.copy()
        headers.pop('Content-Type', None)
        
        response = self.make_request('DELETE', url, timeout=30)
        if response.status_code == 200:
            r = response.json()
            print(f'Deleted {r.get('count')} records successfully')
            return response.json()
        else:
            raise Exception(f'Problem deleting the records given-- further information -- status code -- {response.status_code} -- info -- {response.text}')
        
    """
    Method to delete multiple records from a dataset.
    Raises an error if the provided information is invalid.
    
    Parameters:
        datasetuuid (str): The UUID of the dataset.
        record_uuids (list): A list of record UUID strings to delete.
        user (str): The username of the user performing the action.
    
    Returns:
        dict: Information about the deleted records, including the count and the list of deleted record identifiers.
    """

    def delete_records(self, dataset_uuid: str, record_uuids: list, user_email: str = None) -> Dict:
        user_email = user_email or self.username 
        url = f'{self.base_url}/record'

        json = {'record_uuids': record_uuids}
        print(json)
        headers = self.headers.copy()
        headers.pop('Content-Type', None)
        
        response = self.make_request('DELETE', url, json = json, timeout=30)
        if response.status_code == 200:
            r = response.json()
            print(f'Deleted {r.get('count')} records successfully')
            return response.json()
        else:
            raise Exception(f'Problem deleting the records given-- further information -- status code -- {response.status_code} -- info -- {response.text}')
            
    """
    Method to get all the records of a dataset
    
    Parameters:
        datasetuuid (str): dataset uuid
        limit (int): default is 100 and max is also 100 per request
        offset (int): for pagination where to start
    
    Returns:
        dict - count anad records as keys and value is how many records returned
    """

    def get_dataset(self, dataset_uuid:str, limit: int = 100, offset : int = 0) -> Dict:
        url = f'{self.base_url}/dataset/{dataset_uuid}/{limit}/{offset}'
       # params = {'limit' : limit, 'offset': offset}

        response = self.make_request('GET', url, timeout = 60)

        if response.status_code == 200:
            print(f'Got the dataset records with limit--{limit} and offset -- {offset}, total {response.json().get('count')} records retreived')
            return response.json()
        else:
            raise Exception(f'Problem getting the dataset records look further at status code-- {response.status_code} and more details -- {response.text}')

    """
    Upload a file to ODR using the file-path version.
    Raises an error if any provided information is invalid.
    
    Parameters:
        filepath (str): The local path to the file being uploaded.
        recorduuid (str): The UUID of the record to which the file will be attached.
        datasetuuid (str): The UUID of the dataset containing the record.
        templatefielduuid (str, optional): The UUID of the template field, if applicable.
        fielduuid (str): The UUID of the field to associate the file with.
        name (str, optional): A display name for the file.
        user_email (str, optional): The email of the user performing the upload, used for logging.
    
    Returns:
        dict: Record information including the newly attached file.
    """
    def upload_file_updated(self, file_path: str, record_uuid: str, dataset_uuid: str, 
                   template_field_uuid: str, field_uuid: str = "", 
                   name: str = None, user_email: str = None) -> Dict:
        
        user_email = user_email or self.username
        url = f"{self.base_url}/file"
        
        if not os.path.exists(file_path):
            raise FileNotFoundError(f"File not found: {file_path}")
        
        # Prepare form data
        with open(file_path, 'rb') as f:
            files = {
                'file': (os.path.basename(file_path), f, 'application/octet-stream')
            }
            
            data = {
                'name': name or os.path.basename(file_path),
                'dataset_uuid': dataset_uuid,
                'user_email': user_email,
                'field_uuid': field_uuid,
                'template_field_uuid': template_field_uuid,
                'record_uuid': record_uuid
            }
            
            print(f"📤 Uploading file: {os.path.basename(file_path)}")
            
            # Remove Content-Type header for multipart form data
            headers = self.headers.copy()
            headers.pop('Content-Type', None)
            
            response = requests.post(url, headers=headers, files=files, data=data, timeout=60)
            
            # Handle 401 and retry
            if response.status_code == 401:
                print("🔄 Token expired, refreshing...")
                self.authenticate()
                headers = self.headers.copy()
                headers.pop('Content-Type', None)
                # Need to reopen file
                with open(file_path, 'rb') as f2:
                    files = {'file': (os.path.basename(file_path), f2, 'application/octet-stream')}
                    response = requests.post(url, headers=headers, files=files, data=data, timeout=60)
        
        if response.status_code == 200:
            print(f"✅ File uploaded successfully!")
            return response.json()
        else:
            raise Exception(f"Failed to upload file: {response.status_code} - {response.text}")

        
    """
    Helper function to fetch the ODR template.
    
    May raise an error if the provided information is invalid.
    
    Parameters:
        self: object
    
    Returns:
        dict: A dictionary representing the dataset template, including fields,
              their IDs, and associated metadata.
    """

    def fetch_template(self):
        url = f'{URL}/template/{DATASET_UUID}'
        response = requests.get(url, headers = {'Authorization' : f'Bearer {self.token}'}, timeout = 60)
        response.raise_for_status()
        return response.json()


    """
    Helper function used by discover_tag_options to check tag hierarchy.

    Parameters:
        self: object
        taglist (list): List of tags to evaluate

    Returns:
        dict: A mapping where each key is a tag name and each value is its
          corresponding unique option UUID.
    """

    #poc problem, tag heirarchy problem but this is supposed to help with tag heirarchy    
    def flatten_tags(self, tag_list: list):
        out = {}
        for t in tag_list:
            name = t.get('name') or t.get('tag_name')
            uuid = t.get('template_tag_uuid')
            if name and uuid:
                out[name] = uuid
            for k in ('children', 'tags'):
                if t.get(k):
                    out.update(flatten_tags(t[k]))
        return out
    """
    Discovers the tag options already configured in the ODR by retrieving the template.
    
    Only tags that have been set up in the ODR will appear; any missing tags must be
    configured directly in the ODR. 
    *** I will check with Nate to confirm whether tags can be added via the API.
    
    Parameters:
        self: object
    
    Returns:
        dict: A dictionary where each key is a field name and each value is another
              dictionary containing tag field names and their corresponding UUIDs.
    """
    def discover_tag_options(self):
        #get the template from ODR
        temp = self.fetch_template()
        #set of tag fields
        tag_fields = ('Units Tags', 'Methods Tags', 'Conversions Tags', 'Taxonomies Tags', 'Process Tags')
        #a disctionary of {tag: {tag_name: taguuid}}
        options = {tf: {} for tf in tag_fields}
        #get the fields from template
        for field in temp.get('fields', []):
            #get name of field
            name = field.get('name')
            #if name is a tag then add to dictionary
            if name in tag_fields:
                options[name] = self.flatten_tags(field.get('tags', []))
        #for k,v in options.items():
            #print(f' {k}: {len(v)} options')
        return options
        
    """
    Discovers the radio options already configured in the ODR by retrieving the template.
    
    Radio options that have not been set up in the ODR will not appear and must be
    configured directly in the ODR.
    
    Parameters:
        self: object
    
    Returns:
        dict: A dictionary where each key is a field name and each value is another
              dictionary containing radio option names and their corresponding UUIDs.
    """

    def discover_radio_options(self):
        #getting the template from ODR
        temp = self.fetch_template()
        radio_fields = ['Indicative Class', 'Data Type', 'Sample Type', 'Extant Class', 'Mixed Class']
        options = {rf: {} for rf in radio_fields}
        for field in temp.get('fields', []):
            name = field.get('name')
            if name in radio_fields:
                for ro in field.get('radio_options', []):
                    ro_name = ro.get('name') or ro.get('option_name')
                    ro_uuid = ro.get('template_radio_option_uuid')
                    if ro_name and ro_uuid:
                        options[name][ro_name] = ro_uuid
        #for k, v in options.items():
            #print(f'{k}: has {len(v)} radio options and these are -> {list(v.keys())}')
        return options
    
        #update a record field using new production API settings 
    #/record/{record_uuid}/{field_uuid}/{option_uuid}/{selected|unselected}

    """
    Discovers the UUIDs of field options already configured in the ODR by
    retrieving the associated template. This allows automated and easier
    lookup of option identifiers.
    
    Note:
        Field options that have not been configured in the ODR will not
        appear in the results and must be set up directly within the ODR.
    
    Parameters:
        self: object
    
    Returns:
        dict: A dictionary mapping each field name to its corresponding
              field UUID.
    """

    def discover_field_options(self):
        field_options = {}
        temp = self.fetch_template()
        for field in temp.get('fields', []):
            if field.get('name') not in field_options:
                field_options[field.get('name')] = field.get('field_uuid')
        return field_options
        

    """
    Updates the selected option for a specified field by sending a boolean
    indicating whether the option should be marked as selected.
    
    Note:
        This endpoint was recently added to the documentation by Nate and
        provides a simpler way to update field options. The target field must
        be a radio or tag field.
    
    Parameters:
        self: object
        record_uuid (str): UUID of the record to update
        field_uuid (str): UUID of the field whose option will be updated
        option_uuid (str): UUID of the option (radio, tag, etc.) to select
        selected (bool): True if the option should be selected, False otherwise
    
    Returns:
        dict: A dictionary containing metadata about the database and record,
              including creation and modification details, nested records,
              and associated fields.
    """
    def update_field_option(self, record_uuid : str, field_uuid : str, option_uuid : str, selected : bool):
        if selected:
            selected = 'selected'
            url = f'{self.base_url}/record/{record_uuid}/{field_uuid}/{option_uuid}/{selected}'
        else:
            unselected = 'unselected'
            url = f'{self.base_url}/record/{record_uuid}/{field_uuid}/{option_uuid}/{unselected}'
        headers = self.headers.copy()
        response = requests.put(url, headers = headers)
        if response.status_code == 200:
            print('Field Updated or Changed')
            return response.json
        elif response.status_code == 401:
            self.authenticate()
            response = requests.put(url, headers = headers)
            if response.status_code == 200:
                print('Field Updated or Changed')
                return response.json
        else:
            raise Exception(f'Something went wrong -- status_code -- {response.status_code} -- {response.text}')

    """
    Updates the value of a specified field with the provided input.

    Note:
        This endpoint was recently added to the documentation by Nate and
        provides a simpler way to update field values.

    Parameters:
        self: object
        record_uuid (str): UUID of the record to update
        field_uuid (str): UUID of the field whose value will be updated
        value (str): The new value to assign to the field

    Returns:
        dict: A dictionary containing metadata about the database and record,
              including creation and modification details, nested records,
              and associated fields.
    """

    def update_field_value(self, record_uuid : str, field_uuid : str, value : str):
        url = f'{self.base_url}/record/{record_uuid}/{field_uuid}/value'
        headers = self.headers.copy()
        if value:
            payload = {'value' : value}
            response = requests.post(url, headers = headers, json = payload)
            if response.status_code == 200:
                print('Field Updated or Changed')
                return response.json
            elif response.status_code == 401:
                self.authenticate()
                response = requests.put(url, headers = headers)
                if response.status_code == 200:
                    print('Field Updated or Changed')
                    return response.json
            else:
                raise Exception(f'Something went wrong -- status_code -- {response.status_code} -- {response.text}')


    """
    Check whether a record exists in the database using its unique ID.
    
    Note:
        This endpoint was recently added to the documentation (by Nate) and
        provides a simpler way to search for a specific record.
    
    Parameters:
        self: object
            The class instance.
        dataset_uuid: str
            UUID of the dataset containing the record.
        scobi_record: scobi_raw_record
            The record object whose ID will be checked.
        field_uuid: str
            UUID of the ID field.
        limit: int, optional (default = 0)
            Maximum number of records to return. A value of 0 returns all
            matching records; however, because record IDs are unique, this
            should return either one record (if found) or none.
        offset: int, optional (default = 0)
            Pagination offset.
    
    Returns:
        dict:
            A dictionary containing the record that matches the given ID,
            including its fields and associated metadata. If no record is
            found, an empty dictionary is returned.
    """


    def record_exists(self, dataset_uuid : str, scobi_record , field_uuid : str, limit : int = 0 ,  offset : int = 0, _format : str = 'json'):
        url = f'{self.base_url}/dataset/{dataset_uuid}/search/{limit}/{offset}.{_format}'

        record_id_to_search = scobi_record.id if len(scobi_record.id) < 30 else scobi_record.id[:30]

        payload = { 'fields' : [
                                    {'field_uuid' : field_uuid, 
                                      'value' : record_id_to_search   }
                                ],
                    
            
        }

        response = self.make_request('POST', url = url, json = payload)
        if response.status_code == 200:
            print('Record is being searched')
            return response.json()
        else:
            raise Exception(f'problem searching for the record' -- {response.status_code} -- {response.text})
            

    """
    Upload or update a scobi_raw_record object.

    Parameters:
        self (object): The class instance.
        record_id (str): Unique ID of the record to upload or update.
        radio_options (dict, optional): Available radio-button options defined in the template.
            Defaults to None.
        tag_options (dict, optional): Available tag options defined in the template.
            Defaults to None.
        citation (str, optional): Citation associated with the record. Defaults to None.

    Returns:
        None
    """

    #helper methods to upload_raw_records
    #Function to upload a record fully to ODR it calls the helper funcitons
    def upload_record(self, record_id : str, scobi_record, radio_options: dict = None , tag_options : dict = None , citation : str = None ):
            
            #getting the data file name and converting to csv to save the corresponding data file
            field_uuids = self.discover_field_options()
            data_file_name = (record_id if len(record_id) < 30 else record_id[:30])
            path = '/Users/ametek/Desktop/Project_API/venv/SCOBI/scripts/reocrd_csvs'
            csv_name = f'{record_id}.csv'
            csv_path = os.path.join(path, csv_name)
            scobi_record.data_df.to_csv(csv_path, index = False)
            record_exists = self.record_exists(DATASET_UUID, scobi_record, field_uuids.get('ID'))
            print(record_exists_return)
            
           
            if record_exists:
                odr_record_old = record_exists.get('records')[0]
                odr_record_new = odr_record_old.copy()
                #fields = build_field_payload(self, radio_options, tag_options, citations)
                odr_record_new['fields'] = self.build_field_payload(record_uuid, field_uuids, scobi_record, radio_options, tag_options, citation)
                self.push_record(odr_record_new)
            
            else: 
                record = self.create_record('063c0d3d4bd183ab0dda87c544ae')
            
                record_uuid = record.get('record_uuid')
                #print(f'created record with name: {record.get('record_name')} and uuid is {record_uuid}')
        
                #print(fields)
                #print('pushing metadata fields')
                '''for f in fields:
                #print(f)
                if 'value' in f:
                    value = f.get('value', '')
                elif 'values' in f:
                    value = f.get('values', '')
                elif 'tags' in f:
                    value = ', '.join([tag.get('name', '') for tag in f.get('tags', [])])
                else:
                    value = '?'
                #print(f' Field - {f['field_name']}: value - {str(value)[:50]}...')'''
                record['fields'] = self.build_field_payload( record_uuid, field_uuids, scobi_record, radio_options, tag_options, citation)
                self.push_record(record)
            
            
            #client.upload_file_abdullah(data_file, record_uuid, DATASET_UUID, tfuuid, FIELD_UUIDS['Data File'],data_file_path.name)
            # need to uptade this method
            self.upload_file_updated(str(csv_path), record_uuid, DATASET_UUID, '', field_uuids.get('Data File'),data_file_name)
            #client.upload_file_abdullah(str(source_file_path), record_uuid, DATASET_UUID, tfuuid, FIELD_UUIDS['Source Files'],source_file_path.name)
            #client.upload_file(str(file_path), record_uuid, DATASET_UUID, tfuuid, FIELD_UUIDS['Data File'],file_path.name)
            #client.upload_file(file_path, record_uuid, DATASET_UUID, '', FIELD_UUID['Data File'], file_path.name)
            '''
                self, file_path: str, record_uuid: str, dataset_uuid: str, 
                   template_field_uuid: str, field_uuid: str = "", 
                   name: str = None, user_email: str = None
            '''
            print('File uploaded ')
    
        #method to check if the current record exists in the ODR database

    """
    Build the field payload for a newly created record.

    Parameters:
        self (object): The class instance.
        record_uuid (str): UUID of the record to update.
        field_uuids (dict): Mapping of template field UUIDs.
        scobi_record (scobi_raw_record): The raw scobi record object.
        radio_options (dict, optional): Template radio-button options available
            for use. Defaults to None.
        tag_options (dict, optional): Template tag options available for use.
            Defaults to None.
        citation (str, optional): Citation associated with the record.
            Defaults to None.

    Returns:
        None
    """

    #building the json fields to send to the server
    #check for alternatives this seems brute force
    def build_field_payload(self, record_uuid : str, field_uuids : dict, scobi_record , radio_options : dict = None , tag_options : dict = None , citation : str = None ):
        #in the API the fields are a list of dictionaries, each element of list is a field and it itself has a dictionary
        # the dictionary is key value pair of the different field types and info about them(main parts are the name, field uuid and value
        # the others are generated by API
        #sanity check make sure radio, tag and citation is there if not empty
        radio_options = radio_options or {}
        tag_options = tag_options or {}
        citation = citation or {}
    
        #1 ID(text) - from ID column
        ID = scobi_record.id
        #because ODR ID field is shortvarchar limit ID do we want to rename id to something else like name generator like DELIMIT
        if len(ID) > 30:
            ID = ID[:30]
        if ID:
            value = ID
            #payload = { "value": "123445" }
            #headers = {"Content-Type": "application/json"}
            self.update_field_value(record_uuid, field_uuids.get('ID'), value)
            
        #2build the source field it is a text from source column-- text field
        source_id = scobi_record.source
        if source_id:
            value = source_id
            self.update_field_value(record_uuid, field_uuids.get('Source ID'), value)
            
        #3 Data Type(radio) - from datatype column
        data_type = scobi_record.datatype.capitalize()
        # the datatype in the row file must be in ODR setup if not it will not be marked
        if data_type and data_type in radio_options['Data Type']:
            self.update_field_option(record_uuid, field_uuids.get('Data Type'), radio_options.get('Data Type').get(data_type), True)
        #4 Sample Type(radio option) - from sample columns
        sample = scobi_record.sample
        if sample and sample in radio_options['Sample Type']:
            self.update_field_option(record_uuid, field_uuids.get('Sample Type'), radio_options.get('Sample Type').get(sample), True)
    
        #5 Indicative Class(radio options)- from indclass column
        ind_class = scobi_record.indclass.lower().capitalize()
        if ind_class in radio_options['Indicative Class']:
            self.update_field_option(record_uuid, field_uuids.get('Indicative Class'), radio_options.get('Indicative Class').get(ind_class), True)
        #6 Mixed Class
        mix_class = scobi_record.mixclass.lower().capitalize()
        if mix_class in radio_options['Mixed Class']:
            self.update_field_option(record_uuid, field_uuids.get('Mixed Class'), radio_options.get('Mixed Class').get(mix_class), True)
    
        # Extant Class a radio options from extclass column
        ext_class = scobi_record.extclass.lower().capitalize()
        if ext_class in radio_options['Extant Class']:
            self.update_field_option(record_uuid, field_uuids.get('Extant Class'), radio_options.get('Extant Class').get(ext_class), True)
        # Taxonomies tag(tag field) from taxonomies(semicolon separated)
        tax_tags = scobi_record.taxonomies
        if tax_tags and 'Taxonomies Tags' in tag_options:
            tag_list =[]
            for tax_name in tax_tags:
                    tax_name = tax_name.strip()
                    if tax_name and tax_name in tag_options['Taxonomies Tags']:
                        self.update_field_option(record_uuid, field_uuids.get('Taxonomies Tags'), tag_options.get('Taxonomies Tags').get(tax_name), True)
    
                    #self.update_field_option(record_uuid, field_uuid.get('Taxonomies Tags'), option_uuid.get('Taxonomies Tags').get(tag_name), True)
        # Units tag(tag field) from taxonomies(semicolon separated)
        unit_tag = scobi_record.units
        if unit_tag and 'Units Tags' in tag_options:
            unit_list =[]
            for unit_name in unit_tag:
                    unit_name = unit_name.strip()
                    if unit_name and unit_name in tag_options['Units Tags']:
                        self.update_field_option(record_uuid, field_uuids.get('Units Tags'), tag_options.get('Units Tags').get(unit_name), True)
    
    
        # Methods tag(tag field) from taxonomies(semicolon separated)
        method_tag = scobi_record.methods
        if method_tag and 'Methods Tags' in tag_options:
            method_list =[]
            for method_name in method_tag:
                    method_name = method_name.strip()
                    if method_name and method_name in tag_options['Methods Tags']:
                        self.update_field_option(record_uuid, field_uuids.get('Methods Tags'), tag_options.get('Methods Tags').get(method_name), True)
    
        #Conversions tag(tag field) from taxonomies(semicolon separated)
        # the reason tags are a list compared to radios is because of tag heirarchy
        conv_tag = scobi_record.conversions
        if conv_tag and 'Conversions Tags' in tag_options:
            conv_list =[]
            for conv_name in conv_tag:
                conv_name = conv_name.strip()
                if conv_name and conv_name in tag_options['Conversions Tags']:
                    self.update_field_option(record_uuid, field_uuids.get('Conversions Tags'), tag_options.get('Conversions Tags').get(conv_name), True)
    
        #6. Process tag(tag field) from taxonomies(semicolon separated)
        process_tag = scobi_record.process
        if process_tag and 'Process Tags' in tag_options:
            process_list =[]
            for process_name in process_tag:
                    process_name = process_name.strip()
                    if process_name and process_name in tag_options['Process Tags']:
                        self.update_field_option(record_uuid, field_uuids.get('Process Tags'), tag_options.get('Process Tags').get(process_name), True)
    
         
        # Notes (text) from Notes column
        notes = scobi_record.notes
        if notes:
             value = notes
             self.update_field_value(record_uuid, field_uuids.get('Notes'), value)
        # source links(text) from url column
        url = scobi_record.url
        if url:
            value = url
            self.update_field_value(record_uuid, field_uuids.get('Source Links'), value)
        # Source citation(text) from source_id in scobi.bib files
        citation = citation
        if len(citation) > 250:
            citation = citation[:247] + '...'
        value = citation
        self.update_field_value(record_uuid, field_uuids.get('Source Citation'), value)
    
        #Downloadable (boolean selection) - from downloadable column
        downloadable = scobi_record.downloadable
        if downloadable:
                 self.update_field_option(record_uuid, field_uuids.get('Downloadable?'), option_uuid.get('Downloadable?').get(downloadable), True)


       # def update_field_payload(odr_record, scobi_record):

        #compare odr_record values with scobi_record values, see if there is a change and overwrite it for change

        
  

### Example use cases of the methods written above, uncomment the following to use.

#### Instantiate & Authenticate an object of class client

In [1]:
#client = ODR_Client(URL, USERNAME, PASSWORD)
#client.authenticate()


#### Create empty record

In [2]:
#record_empty = client.create_record(DATASET_UUID)

#check it here: -> https://www.odr.io/063c0d3d4bd183ab0dda87c544ae#/search/display/3121/eyJkdF9pZCI6Ijg4OCJ9/47 (you may need to login in the front end of ODR)
#after logging in check the last record on that page

#print empty record too look at the dictionary response from the server

#print(record_empty)



#### Discover radio & tag options

In [3]:
#discover radio & tag options

#radio_options = client.discover_radio_options()

#print(radio_options)

#tag_options = client.discover_tag_options()

#print(tag_options)

#### Discover Field Options

In [11]:
#fields = client.discover_field_options()
#print(fields)

#### Get a record that was created or already in ODR

In [69]:
#client.get_record('b522c0f65c5418763bdf7e48c224')

#### ODR Template

In [77]:
#see the template of ODR

#template = client.fetch_template()

#print(template.get('name')) #SCOBI Working Files
#print(template.get('fields')) #see what fields exists and what they are
#print(template.get('fields')[0]) #ID field
#print(template.get('fields')[0].get('field_uuid'))